# Week 4 — Improved Model + Final Comparison
Dataset: Daily and Sports Activities (DASA)

## Step 0 — Load data
Upload `daily_and_sports_activities.zip` to Colab left panel, then run this cell.

In [ ]:
import zipfile, os, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import pickle

ZIP_PATH = "/content/daily_and_sports_activities.zip"

if not os.path.isdir("dataset/data"):
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall("dataset")
    print("Unzipped.")
else:
    print("Already unzipped.")

DATA_ROOT = "dataset/data"
X_list, y_list, g_list = [], [], []

for act_folder in sorted(os.listdir(DATA_ROOT)):
    act_path = os.path.join(DATA_ROOT, act_folder)
    if not os.path.isdir(act_path) or not act_folder.startswith("a"):
        continue
    act_id = int(act_folder[1:]) - 1
    for subj_folder in sorted(os.listdir(act_path)):
        subj_path = os.path.join(act_path, subj_folder)
        if not os.path.isdir(subj_path): continue
        subj_id = int(subj_folder[1:])
        for seg_file in sorted(os.listdir(subj_path)):
            if not seg_file.endswith(".txt"): continue
            fpath = os.path.join(subj_path, seg_file)
            try: data = np.loadtxt(fpath, delimiter=",")
            except: continue
            if data.ndim != 2 or data.shape[1] != 45: continue
            X_list.append(data); y_list.append(act_id); g_list.append(subj_id)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
g = np.array(g_list, dtype=np.int64)
print(f"Loaded: X={X.shape}  activities={len(set(y.tolist()))}  subjects={sorted(set(g.tolist()))}")

sp1 = GroupShuffleSplit(1, test_size=0.15, random_state=42)
tv_idx, te_idx = next(sp1.split(X, y, g))
sp2 = GroupShuffleSplit(1, test_size=0.15/0.85, random_state=42)
tr_idx, va_idx = next(sp2.split(X[tv_idx], y[tv_idx], g[tv_idx]))

X_train, y_train = X[tv_idx][tr_idx], y[tv_idx][tr_idx]
X_val,   y_val   = X[tv_idx][va_idx], y[tv_idx][va_idx]
X_test,  y_test  = X[te_idx],          y[te_idx]

N, T, C = X_train.shape
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, C))
def tf(arr):
    n,t,c = arr.shape
    return scaler.transform(arr.reshape(-1,c)).reshape(n,t,c).astype(np.float32)
X_train = tf(X_train); X_val = tf(X_val); X_test = tf(X_test)
print(f"Normalized: mean={X_train.mean():.4f}  std={X_train.std():.4f}")

os.makedirs("data/processed", exist_ok=True)
for name, arr in [("X_train",X_train),("y_train",y_train),
                   ("X_val",X_val),("y_val",y_val),
                   ("X_test",X_test),("y_test",y_test)]:
    np.save(f"data/processed/{name}.npy", arr)
with open("data/processed/scaler.pkl","wb") as f: pickle.dump(scaler, f)
print(f"Done. Train={len(X_train)} Val={len(X_val)} Test={len(X_test)}")


## Step 1 — Imports

In [ ]:
import os, json, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
os.makedirs("results", exist_ok=True)

ACTIVITY_NAMES = [
    "Sitting","Standing","Lying-Back","Lying-Right",
    "Asc-Stairs","Desc-Stairs","Stand-Elev","Move-Elev",
    "Walk-Park","Walk-Flat","Walk-15deg","Run",
    "Stepper","Cross-Trainer","Bike-Horiz",
    "Bike-Vert","Rowing","Jumping","Basketball",
]


## Step 2 — Load processed data

In [ ]:
splits = {}
for k in ("X_train","y_train","X_val","y_val","X_test","y_test"):
    splits[k] = np.load(f"data/processed/{k}.npy")
print(f"Train={splits['X_train'].shape}  Val={splits['X_val'].shape}  Test={splits['X_test'].shape}")


## Step 3 — Data augmentation
Add small noise to training data. Doubles training set.

In [ ]:
rng = np.random.default_rng(42)
noise = rng.normal(0, 0.05, splits["X_train"].shape).astype(np.float32)
X_aug = np.concatenate([splits["X_train"], splits["X_train"]+noise], axis=0)
y_aug = np.concatenate([splits["y_train"], splits["y_train"]], axis=0)
print(f"Original train: {len(splits['X_train'])}  After augmentation: {len(X_aug)}")


## Step 4 — DataLoaders

In [ ]:
def make_loader(X, y, bs=64, shuffle=True):
    Xt = torch.tensor(X.transpose(0,2,1), dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=bs, shuffle=shuffle)

train_loader = make_loader(X_aug,             y_aug,             shuffle=True)
val_loader   = make_loader(splits["X_val"],   splits["y_val"],   shuffle=False)
test_loader  = make_loader(splits["X_test"],  splits["y_test"],  shuffle=False)
print(f"Train={len(train_loader)}  Val={len(val_loader)}  Test={len(test_loader)} batches")


## Step 5 — Improved CNN with residual connections

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch, k=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, k, padding=k//2), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch, ch, k, padding=k//2), nn.BatchNorm1d(ch),
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(self.net(x) + x)

class ImprovedCNN(nn.Module):
    def __init__(self, in_ch=45, num_cls=19):
        super().__init__()
        self.entry = nn.Sequential(
            nn.Conv1d(in_ch, 128, 7, padding=3), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.blocks = nn.Sequential(
            ResBlock(128, 5), nn.MaxPool1d(2),
            ResBlock(128, 3), nn.MaxPool1d(2),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, num_cls),
        )
    def forward(self, x):
        return self.fc(self.pool(self.blocks(self.entry(x))).squeeze(-1))

model = ImprovedCNN().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## Step 6 — Train

In [ ]:
EPOCHS, LR, PATIENCE = 60, 5e-4, 12
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb); loss = criterion(out, yb)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()*len(yb)
            correct    += (out.argmax(1)==yb).sum().item()
            total      += len(yb)
    return total_loss/total, correct/total

history = {"tl":[],"vl":[],"ta":[],"va":[]}
best_vl, patience_cnt, best_w = 1e9, 0, None

print(f"{'Ep':>4} {'TrainLoss':>10} {'TrainAcc':>9} {'ValLoss':>9} {'ValAcc':>8}")
print("-"*46)
for ep in range(1, EPOCHS+1):
    tl,ta = run_epoch(model, train_loader, criterion, optimizer)
    vl,va = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    for k,v in zip(["tl","vl","ta","va"],[tl,vl,ta,va]): history[k].append(v)
    if vl < best_vl:
        best_vl=vl; best_w={k:v.cpu().clone() for k,v in model.state_dict().items()}; patience_cnt=0; mark=" *"
    else:
        patience_cnt+=1; mark=""
    if ep%5==0 or ep==1:
        print(f"{ep:>4} {tl:>10.4f} {ta:>9.4f} {vl:>9.4f} {va:>8.4f}{mark}")
    if patience_cnt>=PATIENCE: print(f"Early stop at epoch {ep}"); break

model.load_state_dict(best_w)
print(f"Best val loss: {best_vl:.4f}")


## Step 7 — Loss and accuracy curves

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
ep = range(1, len(history["tl"])+1)
ax1.plot(ep, history["tl"], label="Train"); ax1.plot(ep, history["vl"], label="Val")
ax1.set_title("Loss — Improved CNN"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, history["ta"], label="Train"); ax2.plot(ep, history["va"], label="Val")
ax2.set_title("Accuracy — Improved CNN"); ax2.legend(); ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("results/week4_improved_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/week4_improved_curves.png")


## Step 8 — Test evaluation

In [ ]:
model.eval()
preds, labels = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        preds.append(model(Xb.to(device)).argmax(1).cpu().numpy())
        labels.append(yb.numpy())
preds  = np.concatenate(preds)
labels = np.concatenate(labels)

imp_acc = accuracy_score(labels, preds)
imp_f1  = f1_score(labels, preds, average="macro")
imp_cm  = confusion_matrix(labels, preds)
print(f"Test accuracy={imp_acc:.4f}  F1={imp_f1:.4f}")
print()
print(classification_report(labels, preds, target_names=ACTIVITY_NAMES, digits=3))


## Step 9 — Final comparison: all 4 models

In [ ]:
def load_json(p):
    if os.path.isfile(p):
        with open(p) as f: return json.load(f)
    return {}

rf   = load_json("results/rf_metrics.json")
cnn  = load_json("results/cnn_metrics.json")
lstm = load_json("results/lstm_metrics.json")

all_models = [
    ("Random Forest", rf.get("test_accuracy",0),   rf.get("test_f1",0)),
    ("1D CNN",        cnn.get("test_accuracy",0),  cnn.get("test_f1",0)),
    ("BiLSTM",        lstm.get("test_accuracy",0), lstm.get("test_f1",0)),
    ("Improved CNN",  imp_acc,                     imp_f1),
]

print(f"{'Model':<18} {'Accuracy':>10} {'F1-macro':>10}")
print("-"*42)
for name, acc, f1 in all_models:
    best = " <-- best" if acc==max(m[1] for m in all_models) else ""
    print(f"{name:<18} {acc:>10.4f} {f1:>10.4f}{best}")


## Step 10 — Final comparison chart

In [ ]:
names = [m[0] for m in all_models]
accs  = [m[1] for m in all_models]
f1s   = [m[2] for m in all_models]
x = np.arange(len(names)); w = 0.35

fig, ax = plt.subplots(figsize=(10,5))
b1 = ax.bar(x-w/2, accs, w, label="Accuracy", color="steelblue")
b2 = ax.bar(x+w/2, f1s,  w, label="F1-macro", color="tomato")
ax.set_title("Final Model Comparison", fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0,1.1); ax.legend(); ax.grid(axis="y",alpha=0.3)
ax.axhline(0.9, color="gray", linestyle="--", linewidth=1)
for bar in list(b1)+list(b2):
    h = bar.get_height()
    if h>0: ax.text(bar.get_x()+bar.get_width()/2, h+0.01, f"{h:.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig("results/week4_final_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/week4_final_comparison.png")


## Step 11 — Error analysis

In [ ]:
per_class = imp_cm.diagonal() / imp_cm.sum(axis=1)
print(f"{'Activity':<22} {'Accuracy':>9}")
print("-"*34)
for name, acc in sorted(zip(ACTIVITY_NAMES, per_class), key=lambda x: x[1]):
    print(f"{name:<22} {acc:>8.3f}  {'#'*int(acc*20)}")

cm_no_diag = imp_cm.copy()
np.fill_diagonal(cm_no_diag, 0)
flat = np.argsort(cm_no_diag.ravel())[::-1][:5]
print("\nTop 5 mistakes:")
for idx in flat:
    r,c = divmod(int(idx),19)
    if cm_no_diag[r,c]>0:
        print(f"  True={ACTIVITY_NAMES[r]:<22} Pred={ACTIVITY_NAMES[c]:<22} ({cm_no_diag[r,c]}x)")


## Step 12 — Save all

In [ ]:
torch.save(model.state_dict(), "results/improved_cnn_model.pth")
np.save("results/improved_cm.npy", imp_cm)
with open("results/improved_history.pkl","wb") as f: pickle.dump(history, f)

imp_metrics = {"test_accuracy":round(float(imp_acc),4),
               "test_f1":      round(float(imp_f1),4),
               "epochs":       len(history["tl"])}
with open("results/improved_metrics.json","w") as f: json.dump(imp_metrics, f, indent=2)

final = {m[0]:{"accuracy":round(float(m[1]),4),"f1":round(float(m[2]),4)} for m in all_models}
with open("results/final_comparison.json","w") as f: json.dump(final, f, indent=2)

print("All saved to results/")
print(json.dumps(final, indent=2))
